# Análisis Académico de Logs de Servidor
Saludos cordiales. En este documento procederemos a examinar un archivo de bitácora arbitrario. En la siguiente celda configuramos las dependencias, el modelo `ApacheLog` y la rutina de carga hacia un DataFrame.

In [ ]:
import pandas as pd
from pydantic import BaseModel, Field, IPvAnyAddress
from datetime import datetime
from typing import Optional
import apachelogs

# Definición del modelo Pydantic para la estructura del log
class ApacheLog(BaseModel):
    ip_address: IPvAnyAddress
    client_identity: Optional[str] = None
    userid: Optional[str] = None
    timestamp: datetime
    method: str
    request_uri: str
    http_version: str
    status_code: int = Field(ge=100, le=599)
    userAgent: Optional[str] = None
    referer: Optional[str] = None
    response_size: Optional[int] = None

def parse_apachelogs(line: str) -> Optional[dict]:
    try:
        log_format = '%h %l %u %t "%r" %s %b "%{Referer}i" "%{User-Agent}i" "%u"'
        parser = apachelogs.LogParser(log_format, errors=None)
        log_line_data = parser.parse(line)
        request_elements = log_line_data.request_line.split()

        # Validamos los datos y retornamos un diccionario para mayor eficiencia en Pandas
        log_obj = ApacheLog(
            ip_address=log_line_data.remote_host,
            client_identity=log_line_data.remote_logname,
            userid=log_line_data.remote_user,
            timestamp=log_line_data.request_time,
            method=request_elements[0] if len(request_elements) > 0 else "-",
            request_uri=request_elements[1] if len(request_elements) > 1 else "-",
            http_version=request_elements[2] if len(request_elements) > 2 else "-",
            status_code=log_line_data.status,
            userAgent=log_line_data.headers_in.get("User-Agent"),
            referer=log_line_data.headers_in.get("Referer"),
            response_size=log_line_data.bytes_sent
        )
        # Convertimos a diccionario (compatible con Pydantic v1 y v2)
        d = log_obj.dict() if hasattr(log_obj, 'dict') else log_obj.model_dump()
        d['ip_address'] = str(d['ip_address']) # Aseguramos formato de cadena
        return d
    except Exception:
        return None

def cargar_datos(ruta: str) -> pd.DataFrame:
    datos = []
    with open(ruta, 'r', encoding='utf-8') as f:
        for linea in f:
            if linea.strip():
                parseado = parse_apachelogs(linea)
                if parseado:
                    datos.append(parseado)
    df = pd.DataFrame(datos)
    if not df.empty:
        df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)
    return df

# Por favor, defina la ruta a su archivo de logs aquí
# df = cargar_datos('ruta/al/access.log')
# df.head()

## 1. Direcciones IP más frecuentes
A continuación, identificaremos las direcciones IP que realizan más peticiones. Espero que esta información facilite su inspección de tráfico.

In [ ]:
# Contamos las ocurrencias de cada IP y mostramos el top 10
top_ips = df['ip_address'].value_counts().head(10)
top_ips

## 2. URLs más solicitadas
Procedemos ahora a extraer los recursos más demandados por los clientes en este conjunto de datos. Revise los resultados.

In [ ]:
# Frecuencia de los identificadores de recursos solicitados
top_urls = df['request_uri'].value_counts().head(10)
top_urls

## 3. Top combinaciones (IP y URL)
Para un análisis más preciso, cruzaremos las IPs con las URLs solicitadas. Esto nos mostrará el interés específico de cada origen.

In [ ]:
# Agrupamos por URI e IP para obtener los pares más comunes
top_combinaciones = df.value_counts(subset=['ip_address', 'request_uri']).head(10)
top_combinaciones

## 4. Direcciones IP con más códigos de error
Filtraremos las respuestas que resultaron en error (códigos 400 en adelante) para aislar las IPs problemáticas. Una disculpa si estos números son altos.

In [ ]:
# Filtramos errores y agrupamos por IP
errores_df = df[df['status_code'] >= 400]
ips_con_errores = errores_df['ip_address'].value_counts().head(10)
ips_con_errores

## 5. URLs con más códigos de error
De la misma manera, buscaremos qué rutas en el servidor generan el mayor número de incidencias o respuestas fallidas.

In [ ]:
# Contamos los URIs dentro del subconjunto de errores
urls_con_errores = errores_df['request_uri'].value_counts().head(10)
urls_con_errores

## 6. URLs con más errores, desglosadas por IP
Calcularemos cuáles son los fallos más comunes cometidos por cada IP específica. Le presento los resultados agrupados.

In [ ]:
# Agrupamos los errores por IP y luego por URL, obteniendo los principales fallos por individuo
errores_por_ip = errores_df.groupby('ip_address')['request_uri'].value_counts()
# Mostramos solo un resumen del top 10 general por brevedad
errores_por_ip.sort_values(ascending=False).head(10)

## 7. Análisis de tráfico por ventana temporal (ej. 10 segundos)
Procederemos a examinar el volumen de peticiones agrupadadas en una ventana de 10 segundos por IP, útil para detectar posibles comportamientos abusivos.

In [ ]:
# Establecemos el timestamp como índice y remuestreamos a ventanas de 10 segundos ('10S')
df_temp = df.set_index('timestamp')
# Agrupamos por IP y ventana de tiempo, contando las peticiones
peticiones_por_ventana = df_temp.groupby('ip_address').resample('10S').size()
# Mostramos los periodos más intensos
top_ventanas = peticiones_por_ventana.nlargest(10)
top_ventanas

## 8. Identificación de solicitudes anormales
Finalmente, intentaremos detectar anomalías. Para este ejercicio, definimos una anomalía como un error de servidor (5xx) o transferencias de datos atípicamente grandes.

In [ ]:
# Calculamos el percentil 99 del tamaño de respuesta para definir peticiones inusualmente pesadas
umbral_tamano = df['response_size'].quantile(0.99) if not df['response_size'].isnull().all() else 0

# Consideramos anormal: Errores internos de servidor o tamaño excesivo
anormales = df[(df['status_code'] >= 500) | (df['response_size'] > umbral_tamano)]

anormales[['ip_address', 'status_code', 'response_size', 'request_uri']].head(10)